# DVC showcase — synthetic-pair end-to-end

Self-contained demo of the `mamba_dvc` v1 pipeline:

1. Build a textured reference volume (band-limited noise).
2. Define an analytical displacement field — a rigid shift composed with a sinusoidal warp.
3. Pull-back warp the reference to produce the deformed volume.
4. Visualize reference, deformed, and the ground-truth field.
5. Run `correlate()` to recover the field from `(reference, deformed)`.
6. Quantify error against the ground truth and render the recovered glyphs.

Volume size is intentionally small (`(96, 128, 128)`, ~6 MB) so each cell completes in seconds and PyVista stays interactive in the notebook.

In [ ]:
from __future__ import annotations

import numpy as np
import pyvista as pv
from mamba_dvc.pipeline.correlate import correlate
from mamba_dvc.types import POIStatus, VoxelSpacing
from mamba_dvc.validate.phantoms import default_phantom
from mamba_dvc.validate.synthetic import compose, make_pair, rigid_shift, sinusoidal
from mamba_dvc.viz._conversion import field_to_polydata
from mamba_dvc.viz.backend import plotter
from mamba_dvc.viz.volume import render_volume

pv.set_jupyter_backend("trame")
pv.OFF_SCREEN = True

p = pv.Plotter(notebook=True, window_size=(400, 300))
p.add_mesh(pv.Sphere(), color="red")
p.show(jupyter_backend="trame")

In [ ]:
import pyvista as pv

pv.set_jupyter_backend("trame")

## 1. Build the synthetic pair

The reference is a bone-like phantom built from analytical primitives: a cylindrical cortical shell along `z`, a perpendicular implant cylinder along `x`, a noise-thresholded trabecular mesh inside the shell, and an off-center spherical defect. A multiplicative texture layer keeps every POI window unique so the FFT NCC has something to lock onto sub-voxel.

Pull-back convention: `deformed(x) = reference(x - u(x))`. `correlate()` recovers the same `u`.

In [ ]:
shape = (96, 128, 128)
spacing = VoxelSpacing((1.0, 1.0, 1.0), "um")

field_fn = compose(
    rigid_shift((1.5, 0.0, -2.0)),
    sinusoidal(amplitude=(0.0, 1.5, 1.5), wavelength=(1.0, 60.0, 60.0)),
)

phantom = default_phantom(shape, seed=0)
pair = make_pair(shape, field_fn, reference=phantom)
reference, deformed = pair.reference, pair.deformed
print(
    f"reference {reference.shape} {reference.dtype}, mean={reference.mean():.3f}, std={reference.std():.3f}"
)
print(
    f"deformed  {deformed.shape}  {deformed.dtype},  mean={deformed.mean():.3f}, std={deformed.std():.3f}"
)

## 2. Reference and deformed, side-by-side

The cortical shell silhouette, the implant cylinder, and the defect cavity are all visible. The deformed view shifts the structure rigidly by `(1.5, 0, -2)` voxels and adds a sinusoidal modulation in `y` and `x`.

In [ ]:
with plotter(shape=(1, 2), window_size=(1024, 512)) as p:
    p.subplot(0, 0)
    p.add_text("reference", font_size=10)
    render_volume(p, reference, spacing=spacing, cmap="bone")
    p.subplot(0, 1)
    p.add_text("deformed", font_size=10)
    render_volume(p, deformed, spacing=spacing, cmap="bone")
    p.link_views()
    p.show(return_viewer=True)

## 3. Ground-truth displacement field

Sample the analytical field on a coarse lattice and draw arrow glyphs over the reference. The `factor` argument inflates glyph length so the warp is visible at this volume size; it does not change the underlying numbers.

In [ ]:
stride = 12
zz, yy, xx = np.meshgrid(
    np.arange(0, shape[0], stride),
    np.arange(0, shape[1], stride),
    np.arange(0, shape[2], stride),
    indexing="ij",
)
gt_coords = np.stack([zz.ravel(), yy.ravel(), xx.ravel()], axis=1).astype(np.float32)
gt_disp = field_fn(gt_coords)

# (z, y, x) -> (x, y, z) for PyVista; spacing is isotropic 1 um here.
gt_points = np.column_stack([gt_coords[:, 2], gt_coords[:, 1], gt_coords[:, 0]]).astype(
    np.float32
)
gt_vectors = np.column_stack([gt_disp[:, 2], gt_disp[:, 1], gt_disp[:, 0]]).astype(np.float32)
gt_poly = pv.PolyData(gt_points)
gt_poly["displacement"] = gt_vectors
gt_poly["magnitude"] = np.linalg.norm(gt_vectors, axis=1)

with plotter(window_size=(900, 700)) as p:
    p.add_text("ground-truth displacement (analytical)", font_size=10)
    render_volume(p, reference, spacing=spacing, cmap="bone")
    p.add_mesh(
        gt_poly.glyph(orient="displacement", scale="magnitude", factor=4.0),
        scalars="magnitude",
        cmap="viridis",
        scalar_bar_args={"title": "||u|| [voxels]"},
    )
    p.show()

## 4. Run the DVC correlator

Single-pass FFT NCC + 3D Gaussian subvoxel fit. With a `(96, 128, 128)` volume and a 32-voxel window this is a few seconds on CPU; CuPy will be far faster once the multi-GPU dispatch lands.

In [ ]:
result = correlate(reference, deformed, window=32, overlap=0.5, search_radius=8)
n_total = result.positions.shape[0]
n_ok = int(result.valid.sum())
print(f"POIs: {n_ok}/{n_total} valid")
for status_value in POIStatus:
    count = int(np.count_nonzero(result.status == status_value))
    if count:
        print(f"  {status_value.name:12s} {count}")

## 5. Error against ground truth

Sample the analytical field at the recovered POI centers and compute per-axis error statistics. The plan budget is sub-0.1 voxel error away from the boundary on textured noise.

In [ ]:
truth = field_fn(result.positions)
valid = result.valid
err_valid = (result.displacements - truth)[valid]

mag_err = np.linalg.norm(err_valid, axis=1)
stats = {
    "MAE_z": float(np.mean(np.abs(err_valid[:, 0]))),
    "MAE_y": float(np.mean(np.abs(err_valid[:, 1]))),
    "MAE_x": float(np.mean(np.abs(err_valid[:, 2]))),
    "RMSE": float(np.sqrt((err_valid**2).mean())),
    "p50": float(np.percentile(mag_err, 50)),
    "p95": float(np.percentile(mag_err, 95)),
    "max": float(mag_err.max()),
}
for k, v in stats.items():
    print(f"  {k:6s} {v:.4f} voxels")

## 6. Recovered field via `viz._conversion.field_to_polydata`

Glyphs colored by displacement magnitude, drawn over the reference. At sub-pixel error this should look indistinguishable from the ground-truth plot in §3.

In [ ]:
poly = field_to_polydata(result, only_valid=True, spacing=spacing)

with plotter(window_size=(900, 700)) as p:
    p.add_text("recovered displacement (correlate())", font_size=10)
    render_volume(p, reference, spacing=spacing, cmap="bone")
    p.add_mesh(
        poly.glyph(orient="displacement", scale="magnitude", factor=4.0),
        scalars="magnitude",
        cmap="viridis",
        scalar_bar_args={"title": "||u|| [um]"},
    )
    p.show()

### Error glyphs (recovered − truth)

Color = error magnitude. Hot spots tend to cluster near the volume boundary — the warp samples outside the reference there, and the Tukey taper deweights those POIs but cannot fix them. The `factor=40` makes the (sub-voxel) errors visible; do not read it as physical scale.

In [ ]:
valid_pos = result.positions[valid]
valid_err = result.displacements[valid] - truth[valid]
err_points = np.column_stack([valid_pos[:, 2], valid_pos[:, 1], valid_pos[:, 0]]).astype(
    np.float32
)
err_vectors = np.column_stack([valid_err[:, 2], valid_err[:, 1], valid_err[:, 0]]).astype(
    np.float32
)
err_poly = pv.PolyData(err_points)
err_poly["error"] = err_vectors
err_poly["magnitude"] = np.linalg.norm(err_vectors, axis=1)

with plotter(window_size=(900, 700)) as p:
    p.add_text("error glyphs (recovered − truth)", font_size=10)
    p.add_mesh(
        err_poly.glyph(orient="error", scale="magnitude", factor=40.0),
        scalars="magnitude",
        cmap="magma",
        scalar_bar_args={"title": "|error| [voxels]"},
    )
    p.show()